In [1]:
import pandas as pd
import numpy as np
from pathlib import Path

PROJECT_ROOT = Path.cwd().parents[1]
DATA_RAW = PROJECT_ROOT / "data" / "raw"
DATA_PROCESSED = PROJECT_ROOT / "data" / "processed"

PROJECT_ROOT

PosixPath('/Users/jeeveshmishra/Desktop/IPO-Success-Prediction/IPO-Success-Prediction')

In [2]:
drhp = pd.read_csv(
    DATA_PROCESSED / "drhp_section_text.csv"
)

drhp.columns.tolist()

['company_name', 'year', 'section', 'text', 'pdf_path', 'text_len']

In [3]:
drhp["company_name_clean"] = (
    drhp["company_name"]
    .str.lower()
    .str.replace(r"[^a-z0-9 ]", "", regex=True)
    .str.replace(r"\s+", " ", regex=True)
    .str.strip()
)

drhp[["company_name", "company_name_clean"]].head()

,company_name,company_name_clean
0,Rajputana Stainless Limited,rajputana stainless limited
1,Rajputana Stainless Limited,rajputana stainless limited
2,Rajputana Stainless Limited,rajputana stainless limited
3,The Promoters Of Our Company Are Bajaj Consume...,the promoters of our company are bajaj consume...
4,The Promoters Of Our Company Are Bajaj Consume...,the promoters of our company are bajaj consume...


In [4]:
ipo_master_raw = pd.read_csv(
    PROJECT_ROOT / "data" / "processed" / "ipo_master_stage1.csv"
)

ipo_master_raw.columns.tolist()

['COMPANY NAME',
 'SECURITY TYPE',
 'ISSUE PRICE',
 'Symbol',
 'ISSUE START DATE',
 'ISSUE END DATE',
 'DATE OF LISTING']

In [5]:
ipo_master = ipo_master_raw.copy()

ipo_master["company_name_clean"] = (
    ipo_master["COMPANY NAME"]
    .str.lower()
    .str.replace(r"[^a-z0-9 ]", "", regex=True)
    .str.replace(r"\s+", " ", regex=True)
    .str.strip()
)

ipo_master["listing_date"] = pd.to_datetime(
    ipo_master["DATE OF LISTING"],
    errors="coerce",
    utc=True
)

ipo_master = ipo_master[
    ["company_name_clean", "listing_date"]
].dropna(subset=["listing_date"])

ipo_master.shape

(1179, 2)

In [6]:
ipo_snapshot = (
    ipo_master
    .sort_values("listing_date")
    .drop_duplicates(subset=["company_name_clean"], keep="first")
)

ipo_snapshot.shape

(987, 2)

In [7]:
drhp_mapped = drhp.merge(
    ipo_snapshot,
    on="company_name_clean",
    how="left",
    validate="many_to_one"
)

drhp_mapped["listing_date"].isna().mean()

np.float64(0.6057007125890737)

In [8]:
drhp_mapped.loc[
    drhp_mapped["listing_date"].isna(),
    "company_name"
].value_counts().head(10)

company_name
L&T Finance Holdings Limited              6
Communications Limited                    6
Sadbhav Infrastructure Project Limited    6
Smc Global Securities Limited             6
Hinduja Leyland Finance Limited           6
Lavasa Corporation Limited                6
Just Dial Limited                         5
Rajputana Stainless Limited               3
Mesh Limited                              3
Renew Pow Er Limited                      3
Name: count, dtype: int64

In [9]:
drhp_valid = drhp_mapped.dropna(subset=["listing_date"]).copy()

drhp_valid.shape

(332, 8)

In [10]:
drhp_valid["company_name_clean"].nunique()

115

In [11]:
drhp_agg = (
    drhp_valid
    .groupby("company_name_clean", as_index=False)
    .agg(
        drhp_total_chars=("text_len", "sum"),
        drhp_sections=("section", "nunique")
    )
)

drhp_agg.describe()

,drhp_total_chars,drhp_sections
count,115.000000,115.000000
mean,57404.573913,2.652174
std,19367.982582,0.496364
min,775.000000,1.000000
25%,40000.000000,2.000000
50%,60000.000000,3.000000
75%,60000.000000,3.000000
max,180000.000000,3.000000


In [12]:
OUT_PATH = PROJECT_ROOT / "data" / "processed"
OUT_PATH.mkdir(parents=True, exist_ok=True)

drhp_agg.to_csv(
    OUT_PATH / "drhp_aggregated_features.csv",
    index=False
)

print("Saved:", OUT_PATH / "drhp_aggregated_features.csv")

Saved: /Users/jeeveshmishra/Desktop/IPO-Success-Prediction/IPO-Success-Prediction/data/processed/drhp_aggregated_features.csv
